In [9]:
import sys
sys.path.append("../src")

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

from machineguard.modeling import build_random_forest_model

In [2]:
DATA_PATH = "../data/ai4i2020.csv"
df = pd.read_csv(DATA_PATH)

df["temperature_difference"] = (
    df["Process temperature [K]"] - df["Air temperature [K]"]
)
df["power_proxy"] = (
    df["Torque [Nm]"] * df["Rotational speed [rpm]"]
)

In [3]:
feature_cols = [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
    "temperature_difference",
    "power_proxy",
]

target_col = "Machine failure"

X = df[feature_cols]
y = df[target_col]

In [4]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=42, stratify=y_train_full
)

In [5]:
def calculate_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

In [ ]:
categorical_features = ["Type"]
numeric_features = [col for col in feature_cols if col!=type]

In [10]:
linear_preprocessing = ColumnTransformer(
    transformers=[
        ("category", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeirc", StandardScaler(), numeric_features)
    ]
)

In [11]:
logistic_model = Pipeline(
    steps=[
        ("preprocessing", linear_preprocessing),
        ("classifier", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42,
        ))
    ]
)

In [12]:
random_forest_model = build_random_forest_model(feature_cols)

In [13]:
tree_preprocessing = ColumnTransformer(
    transformers=[
        ("category", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", "passthrough", numeric_features),
    ]
)

In [14]:
boosting_model = Pipeline(
    steps=[
        ("preprocessing", tree_preprocessing),
        ("classifier", HistGradientBoostingClassifier(
            random_state=42,
            class_weight="balanced",
        )),
    ]
)

In [15]:
logistic_model.fit(X_train, y_train)
random_forest_model.fit(X_train, y_train)
boosting_model.fit(X_train, y_train)

ValueError: could not convert string to float: 'L'

In [ ]:
``